## Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

CATALOG = "workspace"
SCHEMA = "transfermarkt"

data = spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/ml/clustering_features_final")
print("Rows:", data.count())

## k selection: WSSSE and silhouette across k=2 to 10
trainingCost is Spark's built-in WSSSE, available directly off the fitted model's summary. Both metrics are computed for every k in the same loop so they can be read side by side rather than as two separate passes.

In [0]:
import mlflow

mlflow.set_experiment("/Shared/transfermarkt_clustering")

evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="prediction",
                                 metricName="silhouette", distanceMeasure="squaredEuclidean")

results = []
for k in range(2, 11):
    with mlflow.start_run(run_name=f"kmeans_k{k}"):
        kmeans = KMeans(featuresCol="features", predictionCol="prediction", k=k, seed=42)
        model = kmeans.fit(data)
        cost = model.summary.trainingCost
        sil = evaluator.evaluate(model.transform(data))
        results.append((k, cost, sil))

        mlflow.log_param("k", k)
        mlflow.log_metrics({"wssse": cost, "silhouette": sil})

        print(f"k={k}: WSSSE={cost:.1f}, silhouette={sil:.4f}")

## Plot both curves together

In [0]:
import matplotlib.pyplot as plt

ks = [r[0] for r in results]
costs = [r[1] for r in results]
sils = [r[2] for r in results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ks, costs, marker="o")
ax1.set_xlabel("k"); ax1.set_ylabel("WSSSE"); ax1.set_title("Elbow method")

ax2.plot(ks, sils, marker="o", color="orange")
ax2.set_xlabel("k"); ax2.set_ylabel("Silhouette score"); ax2.set_title("Silhouette score")

plt.tight_layout()
plt.show()

## k=2, to test what the "mathematically cleanest" split actually represents
Fit and profiled before being dismissed, not assumed away. Averages are reported in real, unscaled units (goals/assists/yellow per 90, minutes per appearance) rather than the scaled cluster centers, since those are what's actually interpretable.

In [0]:
kmeans_k2 = KMeans(featuresCol="features", predictionCol="cluster_k2", k=2, seed=42)
model_k2 = kmeans_k2.fit(data)
predictions_k2 = model_k2.transform(data)

predictions_k2.groupBy("cluster_k2").agg(
    F.count("*").alias("n"),
    F.round(F.avg("goals_per90"), 3).alias("avg_goals_per90"),
    F.round(F.avg("assists_per90"), 3).alias("avg_assists_per90"),
    F.round(F.avg("yellow_per90"), 3).alias("avg_yellow_per90"),
    F.round(F.avg("minutes_per_appearance"), 1).alias("avg_minutes_per_appearance"),
).orderBy("cluster_k2").show()

## k=4 — the primary model

In [0]:
with mlflow.start_run(run_name="final_model_k4"):
    kmeans_k4 = KMeans(featuresCol="features", predictionCol="cluster", k=4, seed=42)
    model_k4 = kmeans_k4.fit(data)
    predictions_k4 = model_k4.transform(data)

    k4_result = next(r for r in results if r[0] == 4)
    mlflow.log_param("k", 4)
    mlflow.log_metric("wssse", k4_result[1])
    mlflow.log_metric("silhouette", k4_result[2])
    mlflow.spark.log_model(model_k4, "model", dfs_tmpdir=f"/Volumes/{CATALOG}/{SCHEMA}/ml/mlflow_tmp")

predictions_k4.groupBy("cluster").agg(
    F.count("*").alias("n"),
    F.round(F.avg("goals_per90"), 3).alias("avg_goals_per90"),
    F.round(F.avg("assists_per90"), 3).alias("avg_assists_per90"),
    F.round(F.avg("yellow_per90"), 3).alias("avg_yellow_per90"),
    F.round(F.avg("minutes_per_appearance"), 1).alias("avg_minutes_per_appearance"),
).orderBy("cluster").show()

## The real test: does k=4 line up with position, or cut across it?
Clusters that map cleanly onto existing position labels would mean K-Means mostly rediscovered a column that already existed. Clusters that mix positions in a coherent way — e.g. a "high-output" group pulling in both strikers and attacking midfielders — are the actually interesting, defensible finding.

In [0]:
dim_player = spark.table("workspace.gold.dim_player")

predictions_with_position = predictions_k4.join(
    dim_player.select("player_key", "position", "sub_position"), "player_key", "left"
)

predictions_with_position.groupBy("cluster", "position").count() \
    .orderBy("cluster", F.desc("count")).show(40)

## Visualizing the separation
Two panels because no single pair of axes shows all four clusters clearly: goals-vs-assists separates the two higher-output clusters (2, 3) from the two lower-output ones (0, 1), while yellow-vs-minutes is what actually distinguishes 0 from 1.

In [0]:
import matplotlib.pyplot as plt

pdf = predictions_k4.select("goals_per90", "assists_per90", "yellow_per90",
                             "minutes_per_appearance", "cluster").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ["#d62728", "#1f77b4", "#2ca02c", "#ff7f0e"]

cluster_labels = {
    0: "Combative",
    1: "Positional/Reliable",
    2: "Creative",
    3: "Pure Scorer",
}

for c in range(4):
    subset = pdf[pdf["cluster"] == c]
    axes[0].scatter(subset["goals_per90"], subset["assists_per90"], alpha=0.3, s=8, c=colors[c], label=f"{cluster_labels[c]}")
    axes[1].scatter(subset["yellow_per90"], subset["minutes_per_appearance"], alpha=0.3, s=8, c=colors[c], label=f"{cluster_labels[c]}")

axes[0].set_xlabel("goals_per90"); axes[0].set_ylabel("assists_per90")
axes[1].set_xlabel("yellow_per90"); axes[1].set_ylabel("minutes_per_appearance")
axes[0].legend(); axes[1].legend()
plt.tight_layout()
plt.show()

## Write final labeled output
Labels are descriptive shorthand grounded in the profile table above, not arbitrary — worth relabeling if you read them differently, but they should be defensible from the numbers regardless of the exact wording. This table (not the scaled features) is what a dashboard or report would actually query.

In [0]:
cluster_labels = {
    0: "Combative",
    1: "Positional/Reliable",
    2: "Creative",
    3: "Pure Scorer",
}
label_udf = F.udf(lambda c: cluster_labels[c])

player_clusters = predictions_with_position.select(
    "player_key", "position", "sub_position",
    "goals_per90", "assists_per90", "yellow_per90", "minutes_per_appearance",
    F.col("cluster"), label_udf(F.col("cluster")).alias("cluster_label")
)

player_clusters.write.mode("overwrite").parquet(f"/Volumes/{CATALOG}/{SCHEMA}/ml/player_style_clusters")
print("Written:", player_clusters.count(), "rows")
player_clusters.groupBy("cluster_label").count().orderBy(F.desc("count")).show()